In [2]:
# =============================================================================
# 🌽 CORN — PHASE 2: FINAL MASTER IMAGE TESTING
# =============================================================================
# Strategy:
# INTERNAL  = 30 images (10/class) from the already-used verified dataset,
#             specifically VALIDATION images, excluding held-out TEST images.
#
# EXTERNAL  = 30 completely new real-world images (10/class) stored separately
#             in 7th Test_Images/Corn/External real world data.
#
# IMPORTANT:
# The final model was evaluated using raw uint8 [0,255] images.
# Therefore this cell DOES NOT apply MobileNetV2 preprocess_input externally.
# This prevents double preprocessing.
# =============================================================================

import os
import random
import hashlib
import shutil
from pathlib import Path
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from PIL import Image
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow.keras.models import load_model


# =============================================================================
# 1. GOOGLE DRIVE
# =============================================================================

from google.colab import drive
drive.mount('/content/drive')


# =============================================================================
# 2. PROJECT PATHS
# =============================================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Plant Disease Detection (Computer Vision)"
)

INTERNAL_ROOT = PROJECT_ROOT / "1st Raw_Data" / "Corn" / "Final_Combined"

PROCESSED_NPZ = (
    PROJECT_ROOT
    / "3rd Preprocessing"
    / "Corn"
    / "corn_processed_data.npz"
)

SPLIT_MANIFEST = (
    PROJECT_ROOT
    / "3rd Preprocessing"
    / "Corn"
    / "corn_split_manifest.csv"
)

MODEL_PATH = (
    PROJECT_ROOT
    / "6th Trained_Model"
    / "Corn"
    / "corn_mobilenetv2_final.keras"
)

EXTERNAL_ROOT = (
    PROJECT_ROOT
    / "7th Test_Images"
    / "Corn"
    / "External real world data"
)

OUTPUT_ROOT = (
    PROJECT_ROOT
    / "7th Test_Images"
    / "Corn"
    / "Testing_Results"
)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)


# =============================================================================
# 3. CONFIGURATION
# =============================================================================

CLASS_NAMES = [
    "Healthy",
    "Common_Rust",
    "Northern_Leaf_Blight"
]

CLASS_TO_ID = {
    "Healthy": 0,
    "Common_Rust": 1,
    "Northern_Leaf_Blight": 2
}

ID_TO_CLASS = {
    0: "Healthy",
    1: "Common_Rust",
    2: "Northern_Leaf_Blight"
}

IMAGE_SIZE = (224, 224)
IMAGES_PER_CLASS = 10

VALID_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp", ".webp"
}

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


print("=" * 90)
print("🌽 CORN — PHASE 2: FINAL MASTER IMAGE TESTING")
print("=" * 90)

print("\nInternal source:")
print(INTERNAL_ROOT)

print("\nExternal source:")
print(EXTERNAL_ROOT)

print("\nModel:")
print(MODEL_PATH)

print("\nTesting output:")
print(OUTPUT_ROOT)


# =============================================================================
# 4. CHECK REQUIRED FILES / FOLDERS
# =============================================================================

print("\n" + "=" * 90)
print("CHECKING REQUIRED SOURCES")
print("=" * 90)

checks = [
    ("Final trained model", MODEL_PATH.is_file()),
    ("Processed NPZ", PROCESSED_NPZ.is_file()),
    ("Split manifest", SPLIT_MANIFEST.is_file()),
    ("Internal Final_Combined dataset", INTERNAL_ROOT.is_dir()),
    ("External real-world dataset", EXTERNAL_ROOT.is_dir()),
]

all_sources_ok = True

for name, status in checks:
    print(("PASS" if status else "FAIL") + f": {name} found" if status
          else f"FAIL: {name} not found")
    all_sources_ok = all_sources_ok and status

if not all_sources_ok:
    raise RuntimeError(
        "STOP: One or more required testing sources are missing."
    )


# =============================================================================
# 5. LOAD FINAL MODEL
# =============================================================================

print("\n" + "=" * 90)
print("LOADING FINAL CORN MODEL")
print("=" * 90)

model = load_model(MODEL_PATH)

print("PASS: Model loaded successfully")
print("Input shape :", model.input_shape)
print("Output shape:", model.output_shape)

if model.output_shape[-1] != 3:
    raise RuntimeError("STOP: Model does not have exactly 3 output classes.")

print("PASS: Model has exactly 3 output classes")


# =============================================================================
# 6. LOAD PROCESSED DATA
# =============================================================================

print("\n" + "=" * 90)
print("LOADING PROCESSED DATA")
print("=" * 90)

data = np.load(PROCESSED_NPZ, allow_pickle=True)

X_test = data["X_test"]
y_test = data["y_test"]

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

if len(X_test) != 270:
    raise RuntimeError(
        f"STOP: Expected 270 held-out test images, found {len(X_test)}."
    )

print("PASS: Held-out test set contains 270 images")


# =============================================================================
# 7. LOAD SPLIT MANIFEST
# =============================================================================

print("\n" + "=" * 90)
print("LOADING SPLIT MANIFEST")
print("=" * 90)

manifest = pd.read_csv(SPLIT_MANIFEST)

print("Manifest rows:", len(manifest))
print("Manifest columns:", list(manifest.columns))

required_manifest_columns = {
    "filepath",
    "class",
    "label",
    "split"
}

if not required_manifest_columns.issubset(manifest.columns):
    raise RuntimeError(
        "STOP: Required split manifest columns are missing."
    )

print("PASS: Required manifest columns found")


# =============================================================================
# 8. BUILD INTERNAL TESTING POOL
# =============================================================================
# IMPORTANT:
# We use VALIDATION images for internal testing.
#
# Why?
# - Training images were used during model learning.
# - Held-out TEST images are reserved for formal evaluation.
# - Validation images provide a separate internal image-testing pool.
# =============================================================================

print("\n" + "=" * 90)
print("BUILDING INTERNAL TESTING POOL")
print("=" * 90)

internal_records = []

for class_name in CLASS_NAMES:

    class_rows = manifest[
        (manifest["class"] == class_name) &
        (manifest["split"].str.lower() == "validation")
    ].copy()

    if len(class_rows) < IMAGES_PER_CLASS:
        raise RuntimeError(
            f"STOP: Not enough validation images for {class_name}. "
            f"Available = {len(class_rows)}, required = {IMAGES_PER_CLASS}"
        )

    # Deterministic selection
    class_rows = class_rows.sort_values("filepath").reset_index(drop=True)

    selected = class_rows.iloc[:IMAGES_PER_CLASS]

    for _, row in selected.iterrows():

        original_path = Path(str(row["filepath"]))

        # Handle both absolute and relative manifest paths
        if original_path.is_absolute() and original_path.exists():
            image_path = original_path
        else:
            possible_paths = [
                INTERNAL_ROOT / original_path,
                PROJECT_ROOT / original_path,
                Path(str(original_path))
            ]

            image_path = None

            for p in possible_paths:
                if p.exists():
                    image_path = p
                    break

        if image_path is None:
            raise RuntimeError(
                f"STOP: Internal image not found:\n{row['filepath']}"
            )

        internal_records.append({
            "testing_type": "Internal",
            "filepath": str(image_path),
            "true_class": class_name,
            "true_label": CLASS_TO_ID[class_name],
            "split_source": "validation"
        })

    print(
        f"{class_name}: "
        f"{len(selected)} validation images selected"
    )

internal_df = pd.DataFrame(internal_records)

if len(internal_df) != 30:
    raise RuntimeError(
        f"STOP: Internal testing pool must contain 30 images. "
        f"Found {len(internal_df)}."
    )

print("\nPASS: Internal testing pool ready")
print("Internal images:", len(internal_df))


# =============================================================================
# 9. INTERNAL TESTING POOL VERIFICATION
# =============================================================================

print("\n" + "=" * 90)
print("VERIFYING INTERNAL TESTING POOL")
print("=" * 90)

internal_class_counts = (
    internal_df["true_class"]
    .value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
)

print(internal_class_counts)

if not all(
    internal_class_counts[class_name] == IMAGES_PER_CLASS
    for class_name in CLASS_NAMES
):
    raise RuntimeError(
        "STOP: Internal testing class balance is incorrect."
    )

if internal_df["filepath"].duplicated().any():
    raise RuntimeError(
        "STOP: Duplicate internal image paths detected."
    )

print("PASS: Exactly 10 internal images per class")
print("PASS: No duplicate internal image paths")
print("PASS: All 3 internal classes present")


# =============================================================================
# 10. VERIFY INTERNAL IMAGES ARE NOT HELD-OUT TEST IMAGES
# =============================================================================

print("\n" + "=" * 90)
print("CHECKING INTERNAL DATA LEAKAGE")
print("=" * 90)

test_manifest = manifest[
    manifest["split"].str.lower() == "test"
].copy()

test_paths = set()

for _, row in test_manifest.iterrows():

    p = Path(str(row["filepath"]))

    possible_paths = [
        p,
        INTERNAL_ROOT / p,
        PROJECT_ROOT / p
    ]

    for candidate in possible_paths:
        if candidate.exists():
            test_paths.add(str(candidate.resolve()))

internal_test_overlap = []

for path in internal_df["filepath"]:

    resolved = str(Path(path).resolve())

    if resolved in test_paths:
        internal_test_overlap.append(path)

if len(internal_test_overlap) > 0:
    raise RuntimeError(
        "STOP: Held-out test image detected in internal testing pool."
    )

print("PASS: No held-out test images reused")


# =============================================================================
# 11. BUILD EXTERNAL REAL-WORLD TESTING POOL
# =============================================================================

print("\n" + "=" * 90)
print("BUILDING EXTERNAL REAL-WORLD TESTING POOL")
print("=" * 90)

external_records = []

for class_name in CLASS_NAMES:

    class_folder = EXTERNAL_ROOT / class_name

    if not class_folder.is_dir():
        raise RuntimeError(
            f"STOP: External class folder missing:\n{class_folder}"
        )

    files = sorted([
        p for p in class_folder.iterdir()
        if p.is_file()
        and p.suffix.lower() in VALID_EXTENSIONS
    ])

    print(f"{class_name}: {len(files)} available images")

    if len(files) < IMAGES_PER_CLASS:
        raise RuntimeError(
            f"STOP: External {class_name} requires "
            f"{IMAGES_PER_CLASS} images."
        )

    # Deterministic selection
    selected_files = files[:IMAGES_PER_CLASS]

    for image_path in selected_files:

        external_records.append({
            "testing_type": "External",
            "filepath": str(image_path),
            "true_class": class_name,
            "true_label": CLASS_TO_ID[class_name],
            "split_source": "new_real_world_data"
        })

    print(
        f"{class_name}: "
        f"{len(selected_files)} new real-world images selected"
    )

external_df = pd.DataFrame(external_records)

if len(external_df) != 30:
    raise RuntimeError(
        f"STOP: External testing pool must contain 30 images. "
        f"Found {len(external_df)}."
    )

print("\nPASS: External testing pool ready")
print("External images:", len(external_df))


# =============================================================================
# 12. EXTERNAL TESTING POOL VERIFICATION
# =============================================================================

print("\n" + "=" * 90)
print("VERIFYING EXTERNAL TESTING POOL")
print("=" * 90)

external_class_counts = (
    external_df["true_class"]
    .value_counts()
    .reindex(CLASS_NAMES, fill_value=0)
)

print(external_class_counts)

if not all(
    external_class_counts[class_name] == IMAGES_PER_CLASS
    for class_name in CLASS_NAMES
):
    raise RuntimeError(
        "STOP: External testing class balance is incorrect."
    )

if external_df["filepath"].duplicated().any():
    raise RuntimeError(
        "STOP: Duplicate external image paths detected."
    )

print("PASS: Exactly 10 external images per class")
print("PASS: No duplicate external image paths")
print("PASS: All 3 external classes present")


# =============================================================================
# 13. IMAGE LOADING FUNCTION
# =============================================================================
# CRITICAL:
#
# The final model was evaluated with X_test stored as:
#     uint8 [0,255]
#
# The model evaluation achieved:
#     97.41% accuracy
#
# Therefore, image testing must reproduce the SAME input pipeline.
#
# We resize + RGB convert + store as uint8.
#
# DO NOT call:
#     tf.keras.applications.mobilenet_v2.preprocess_input()
#
# here, because doing so would preprocess the image before passing it
# to a model whose training/evaluation pipeline already handles the
# MobileNetV2 preprocessing.
# =============================================================================

def load_image_for_model(image_path):

    try:

        image = Image.open(image_path).convert("RGB")

        image = image.resize(
            IMAGE_SIZE,
            Image.Resampling.BILINEAR
        )

        image_array = np.asarray(
            image,
            dtype=np.uint8
        )

        if image_array.shape != (224, 224, 3):
            raise ValueError(
                f"Unexpected image shape: {image_array.shape}"
            )

        return image_array

    except Exception as e:

        raise RuntimeError(
            f"Could not process image:\n{image_path}\nError: {e}"
        )


# =============================================================================
# 14. VERIFY SAMPLE INPUT
# =============================================================================

print("\n" + "=" * 90)
print("VERIFYING IMAGE INPUT PIPELINE")
print("=" * 90)

sample_path = internal_df.iloc[0]["filepath"]

sample_array = load_image_for_model(sample_path)

print("Sample image:", Path(sample_path).name)
print("Prepared shape:", sample_array.shape)
print(
    "Prepared range:",
    sample_array.min(),
    "to",
    sample_array.max()
)
print("Prepared dtype:", sample_array.dtype)

if sample_array.shape != (224, 224, 3):
    raise RuntimeError(
        "STOP: Image preprocessing shape is incorrect."
    )

if sample_array.dtype != np.uint8:
    raise RuntimeError(
        "STOP: Image dtype is not uint8."
    )

print("PASS: 224 × 224 × 3 input verified")
print("PASS: RGB conversion verified")
print("PASS: uint8 [0,255] input verified")
print("PASS: No external MobileNetV2 preprocessing applied")


# =============================================================================
# 15. PREDICTION FUNCTION
# =============================================================================

def predict_dataset(df, dataset_name):

    print("\n" + "=" * 90)
    print(f"{dataset_name.upper()} — PREDICTIONS")
    print("=" * 90)

    results = []

    for _, row in tqdm(
        df.iterrows(),
        total=len(df),
        desc=f"{dataset_name} images"
    ):

        image_path = row["filepath"]
        true_class = row["true_class"]
        true_label = int(row["true_label"])

        image_array = load_image_for_model(image_path)

        batch = np.expand_dims(
            image_array,
            axis=0
        )

        # IMPORTANT:
        # Raw uint8 [0,255] input is passed directly to the final model.
        prediction = model.predict(
            batch,
            verbose=0
        )[0]

        predicted_label = int(
            np.argmax(prediction)
        )

        predicted_class = ID_TO_CLASS[predicted_label]

        confidence = float(
            prediction[predicted_label]
        )

        correct = (
            predicted_label == true_label
        )

        results.append({
            "testing_type": dataset_name,
            "filepath": image_path,
            "true_class": true_class,
            "true_label": true_label,
            "predicted_class": predicted_class,
            "predicted_label": predicted_label,
            "confidence": confidence,
            "confidence_percent": confidence * 100,
            "correct": correct
        })

    result_df = pd.DataFrame(results)

    print(
        f"PASS: {dataset_name} predictions generated"
    )

    return result_df


# =============================================================================
# 16. RUN INTERNAL TESTING
# =============================================================================

internal_results = predict_dataset(
    internal_df,
    "Internal"
)


# =============================================================================
# 17. RUN EXTERNAL TESTING
# =============================================================================

external_results = predict_dataset(
    external_df,
    "External"
)


# =============================================================================
# 18. RESULT SUMMARY FUNCTION
# =============================================================================

def create_summary(result_df, testing_type):

    total = len(result_df)

    correct = int(
        result_df["correct"].sum()
    )

    incorrect = total - correct

    accuracy = (
        correct / total
        if total > 0
        else 0
    )

    average_confidence = (
        result_df["confidence"].mean()
        if total > 0
        else 0
    )

    return {
        "Testing_Type": testing_type,
        "Total_Images": total,
        "Correct": correct,
        "Incorrect": incorrect,
        "Accuracy_Percent": accuracy * 100,
        "Average_Confidence_Percent":
            average_confidence * 100
    }


# =============================================================================
# 19. INTERNAL RESULTS
# =============================================================================

print("\n" + "=" * 90)
print("🌽 INTERNAL TESTING RESULTS")
print("=" * 90)

internal_total = len(internal_results)
internal_correct = int(
    internal_results["correct"].sum()
)

internal_accuracy = (
    internal_correct / internal_total
) * 100

internal_confidence = (
    internal_results["confidence"].mean()
) * 100

print("Total images       :", internal_total)
print("Correct predictions:", internal_correct)
print("Incorrect          :", internal_total - internal_correct)
print(f"Accuracy           : {internal_accuracy:.2f}%")
print(f"Average confidence : {internal_confidence:.2f}%")


print("\n" + "-" * 90)
print("INTERNAL PER-CLASS PERFORMANCE")
print("-" * 90)

internal_class_results = []

for class_name in CLASS_NAMES:

    subset = internal_results[
        internal_results["true_class"] == class_name
    ]

    total = len(subset)
    correct = int(subset["correct"].sum())

    accuracy = (
        correct / total * 100
        if total > 0
        else 0
    )

    avg_confidence = (
        subset["confidence"].mean() * 100
        if total > 0
        else 0
    )

    print(
        f"{class_name}: "
        f"{correct}/{total} correct = "
        f"{accuracy:.2f}%"
    )

    internal_class_results.append({
        "Testing_Type": "Internal",
        "Class": class_name,
        "Total_Images": total,
        "Correct": correct,
        "Incorrect": total - correct,
        "Accuracy_Percent": accuracy,
        "Average_Confidence_Percent":
            avg_confidence
    })

internal_class_df = pd.DataFrame(
    internal_class_results
)


# =============================================================================
# 20. EXTERNAL RESULTS
# =============================================================================

print("\n" + "=" * 90)
print("🌍 EXTERNAL REAL-WORLD TESTING RESULTS")
print("=" * 90)

external_total = len(external_results)
external_correct = int(
    external_results["correct"].sum()
)

external_accuracy = (
    external_correct / external_total
) * 100

external_confidence = (
    external_results["confidence"].mean()
) * 100

print("Total images       :", external_total)
print("Correct predictions:", external_correct)
print("Incorrect          :", external_total - external_correct)
print(f"Accuracy           : {external_accuracy:.2f}%")
print(f"Average confidence : {external_confidence:.2f}%")


print("\n" + "-" * 90)
print("EXTERNAL PER-CLASS PERFORMANCE")
print("-" * 90)

external_class_results = []

for class_name in CLASS_NAMES:

    subset = external_results[
        external_results["true_class"] == class_name
    ]

    total = len(subset)
    correct = int(subset["correct"].sum())

    accuracy = (
        correct / total * 100
        if total > 0
        else 0
    )

    avg_confidence = (
        subset["confidence"].mean() * 100
        if total > 0
        else 0
    )

    print(
        f"{class_name}: "
        f"{correct}/{total} correct = "
        f"{accuracy:.2f}%"
    )

    external_class_results.append({
        "Testing_Type": "External",
        "Class": class_name,
        "Total_Images": total,
        "Correct": correct,
        "Incorrect": total - correct,
        "Accuracy_Percent": accuracy,
        "Average_Confidence_Percent":
            avg_confidence
    })

external_class_df = pd.DataFrame(
    external_class_results
)


# =============================================================================
# 21. INTERNAL vs EXTERNAL COMPARISON
# =============================================================================

print("\n" + "=" * 90)
print("📊 INTERNAL vs EXTERNAL COMPARISON")
print("=" * 90)

comparison_df = pd.DataFrame([
    create_summary(
        internal_results,
        "Internal"
    ),
    create_summary(
        external_results,
        "External"
    )
])

print(comparison_df.to_string(index=False))

accuracy_difference = (
    internal_accuracy - external_accuracy
)

print(
    f"\nInternal − External accuracy difference: "
    f"{accuracy_difference:.2f} percentage points"
)


# =============================================================================
# 22. ERROR ANALYSIS
# =============================================================================

print("\n" + "=" * 90)
print("ERROR ANALYSIS")
print("=" * 90)

combined_results = pd.concat(
    [
        internal_results,
        external_results
    ],
    ignore_index=True
)

errors_df = combined_results[
    combined_results["correct"] == False
].copy()

print(
    "Total incorrect predictions:",
    len(errors_df)
)

if len(errors_df) > 0:

    error_summary = (
        errors_df
        .groupby(
            [
                "testing_type",
                "true_class",
                "predicted_class"
            ]
        )
        .size()
        .reset_index(name="count")
    )

    print("\nActual → Predicted errors:")
    print(error_summary.to_string(index=False))

else:

    error_summary = pd.DataFrame(
        columns=[
            "testing_type",
            "true_class",
            "predicted_class",
            "count"
        ]
    )

    print("No incorrect predictions.")


# =============================================================================
# 23. CONFIDENCE ANALYSIS
# =============================================================================

print("\n" + "=" * 90)
print("CONFIDENCE ANALYSIS")
print("=" * 90)

overall_confidence = (
    combined_results["confidence"].mean()
) * 100

correct_confidence = (
    combined_results[
        combined_results["correct"] == True
    ]["confidence"].mean()
) * 100

incorrect_confidence = (
    combined_results[
        combined_results["correct"] == False
    ]["confidence"].mean()
) * 100 if len(errors_df) > 0 else np.nan

print(
    f"Average confidence           : "
    f"{overall_confidence:.2f}%"
)

print(
    f"Correct prediction average   : "
    f"{correct_confidence:.2f}%"
)

if not np.isnan(incorrect_confidence):

    print(
        f"Incorrect prediction average : "
        f"{incorrect_confidence:.2f}%"
    )

else:

    print(
        "Incorrect prediction average : N/A"
    )


# =============================================================================
# 24. SAVE PREDICTION RESULTS
# =============================================================================

print("\n" + "=" * 90)
print("SAVING TESTING RESULTS")
print("=" * 90)

internal_csv = (
    OUTPUT_ROOT /
    "corn_internal_predictions.csv"
)

external_csv = (
    OUTPUT_ROOT /
    "corn_external_predictions.csv"
)

combined_csv = (
    OUTPUT_ROOT /
    "corn_combined_predictions.csv"
)

internal_class_csv = (
    OUTPUT_ROOT /
    "corn_internal_class_results.csv"
)

external_class_csv = (
    OUTPUT_ROOT /
    "corn_external_class_results.csv"
)

comparison_csv = (
    OUTPUT_ROOT /
    "corn_internal_external_comparison.csv"
)

error_csv = (
    OUTPUT_ROOT /
    "corn_testing_error_analysis.csv"
)

summary_csv = (
    OUTPUT_ROOT /
    "corn_final_testing_summary.csv"
)

internal_results.to_csv(
    internal_csv,
    index=False
)

external_results.to_csv(
    external_csv,
    index=False
)

combined_results.to_csv(
    combined_csv,
    index=False
)

internal_class_df.to_csv(
    internal_class_csv,
    index=False
)

external_class_df.to_csv(
    external_class_csv,
    index=False
)

comparison_df.to_csv(
    comparison_csv,
    index=False
)

error_summary.to_csv(
    error_csv,
    index=False
)

final_summary = pd.DataFrame([
    {
        "Testing_Type": "Internal",
        "Total_Images": internal_total,
        "Correct": internal_correct,
        "Incorrect": internal_total - internal_correct,
        "Accuracy_Percent": internal_accuracy,
        "Average_Confidence_Percent": internal_confidence
    },
    {
        "Testing_Type": "External",
        "Total_Images": external_total,
        "Correct": external_correct,
        "Incorrect": external_total - external_correct,
        "Accuracy_Percent": external_accuracy,
        "Average_Confidence_Percent": external_confidence
    }
])

final_summary.to_csv(
    summary_csv,
    index=False
)

print("PASS: Internal prediction CSV saved")
print("PASS: External prediction CSV saved")
print("PASS: Combined prediction CSV saved")
print("PASS: Internal class results saved")
print("PASS: External class results saved")
print("PASS: Comparison results saved")
print("PASS: Error analysis saved")
print("PASS: Final testing summary saved")


# =============================================================================
# 25. CREATE PREDICTION VISUALIZATIONS
# =============================================================================

print("\n" + "=" * 90)
print("CREATING PREDICTION VISUALIZATIONS")
print("=" * 90)


def create_prediction_grid(
    result_df,
    title,
    output_path
):

    fig, axes = plt.subplots(
        5,
        6,
        figsize=(18, 15)
    )

    axes = axes.flatten()

    for i, (_, row) in enumerate(
        result_df.iterrows()
    ):

        image = Image.open(
            row["filepath"]
        ).convert("RGB")

        axes[i].imshow(image)

        true_class = row["true_class"]
        predicted_class = row["predicted_class"]
        confidence = row["confidence_percent"]
        correct = row["correct"]

        status = (
            "CORRECT"
            if correct
            else "INCORRECT"
        )

        axes[i].set_title(
            f"True: {true_class}\n"
            f"Pred: {predicted_class}\n"
            f"{confidence:.1f}% | {status}",
            fontsize=8
        )

        axes[i].axis("off")

    for j in range(
        len(result_df),
        len(axes)
    ):
        axes[j].axis("off")

    fig.suptitle(
        title,
        fontsize=18
    )

    plt.tight_layout()

    plt.savefig(
        output_path,
        dpi=200,
        bbox_inches="tight"
    )

    plt.show()

    plt.close()


internal_visualization = (
    OUTPUT_ROOT /
    "corn_internal_predictions_visualization.png"
)

external_visualization = (
    OUTPUT_ROOT /
    "corn_external_predictions_visualization.png"
)

create_prediction_grid(
    internal_results,
    "Corn — Internal Image Testing",
    internal_visualization
)

print(
    "PASS: Internal prediction visualization saved"
)

create_prediction_grid(
    external_results,
    "Corn — External Real-World Image Testing",
    external_visualization
)

print(
    "PASS: External prediction visualization saved"
)


# =============================================================================
# 26. FINAL MASTER AUDIT
# =============================================================================

print("\n" + "=" * 90)
print("🌽 CORN — FINAL IMAGE TESTING MASTER AUDIT")
print("=" * 90)

audit_checks = {
    "Final model loaded":
        model is not None,

    "Internal source found":
        INTERNAL_ROOT.is_dir(),

    "External source found":
        EXTERNAL_ROOT.is_dir(),

    "Internal testing = 30 images":
        len(internal_results) == 30,

    "External testing = 30 images":
        len(external_results) == 30,

    "Internal has 10 images/class":
        all(
            internal_class_counts == 10
        ),

    "External has 10 images/class":
        all(
            external_class_counts == 10
        ),

    "All 3 classes tested internally":
        set(internal_results["true_class"]) ==
        set(CLASS_NAMES),

    "All 3 classes tested externally":
        set(external_results["true_class"]) ==
        set(CLASS_NAMES),

    "Internal predictions generated":
        len(internal_results) == 30,

    "External predictions generated":
        len(external_results) == 30,

    "Combined results saved":
        combined_csv.is_file(),

    "Internal results saved":
        internal_csv.is_file(),

    "External results saved":
        external_csv.is_file(),

    "No held-out test images reused":
        len(internal_test_overlap) == 0,

    "Input pipeline = uint8 [0,255]":
        sample_array.dtype == np.uint8,

    "Input shape = 224x224x3":
        sample_array.shape == (224, 224, 3)
}

all_audit_pass = True

for check_name, check_result in audit_checks.items():

    if check_result:

        print(
            f"PASS | {check_name}"
        )

    else:

        print(
            f"FAIL | {check_name}"
        )

        all_audit_pass = False


# =============================================================================
# 27. FINAL RESULT
# =============================================================================

print("\n" + "=" * 90)

if all_audit_pass:

    print("🎉 CORN — FINAL IMAGE TESTING: PASS")

    print("\nINTERNAL TESTING")
    print("-" * 50)
    print(
        f"Accuracy: "
        f"{internal_accuracy:.2f}% "
        f"({internal_correct}/{internal_total})"
    )

    print(
        f"Average Confidence: "
        f"{internal_confidence:.2f}%"
    )

    print("\nEXTERNAL REAL-WORLD TESTING")
    print("-" * 50)
    print(
        f"Accuracy: "
        f"{external_accuracy:.2f}% "
        f"({external_correct}/{external_total})"
    )

    print(
        f"Average Confidence: "
        f"{external_confidence:.2f}%"
    )

    print("\nINTERNAL vs EXTERNAL")
    print("-" * 50)

    print(
        f"Internal Accuracy : "
        f"{internal_accuracy:.2f}%"
    )

    print(
        f"External Accuracy : "
        f"{external_accuracy:.2f}%"
    )

    print(
        f"Accuracy Difference: "
        f"{accuracy_difference:.2f} percentage points"
    )

    print("\nTesting results saved to:")
    print(OUTPUT_ROOT)

else:

    print("❌ CORN — FINAL IMAGE TESTING: FAIL")
    print("STOP: One or more audit checks failed.")


print("=" * 90)

Output hidden; open in https://colab.research.google.com to view.